In [ ]:
KE_SRC = 'import os, glob, time, gc\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\nos.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nos.environ.setdefault(\'HF_HUB_DISABLE_TELEMETRY\', \'1\')\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\nIMG = 336\nCROP_MM = 140.0\nSLOTS = [(\'Sagittal\', 1, 18), (\'Sagittal\', 0, 14), (\'Coronal\', 1, 12), (\'Coronal\', 0, 8), (\'Axial\', -1, 12)]\nMAXS = sum((s[2] for s in SLOTS))\nK_EVAL = 42\nNORM = \'imagenet\'\nLAB = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\nARMS = [{\'file\': \'raptor_ft_coatnet_v5_full_swa.pt\', \'arch\': \'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', \'res\': 384, \'w\': 1.0}]\n\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith((\'maxvit\', \'maxxvit\', \'coatnet\', \'coat_\', \'convnext\'))\n    is_vit = not hybrid and any((k in arch for k in (\'vit\', \'deit\', \'dinov2\', \'eva\', \'beit\')))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool=\'token\', dynamic_img_size=True)\n    else:\n        kw.update(global_pool=\'avg\')\n    return timm.create_model(arch, **kw)\n\nclass RaptorClassifier(nn.Module):\n\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum(\'bkn,bkf->bnf\', a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    ck = torch.load(pt_path, map_location=\'cpu\', weights_only=False)\n    arch = ck.get(\'arch\', arch_default)\n    ck_res = int(ck.get(\'res\', res_default))\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck[\'model\'], strict=True)\n    model.eval().to(device)\n    del ck\n    gc.collect()\n    return (model, ck_res)\n\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = (int(valid.min()), int(valid.max()))\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\ndef eval_windows(vol, mask, k, res, norm=NORM):\n    D = vol.shape[0]\n    cs = _eval_centers(mask, D, k)\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n        t = torch.from_numpy(tri)\n        if t.shape[-1] != res:\n            t = F.interpolate(t[None], size=(res, res), mode=\'bilinear\', align_corners=False)[0]\n        wins[j] = t.numpy()\n    x = torch.from_numpy(wins)\n    if norm == \'imagenet\':\n        x = (x - _MEAN) / _STD\n    return x\n\n@torch.no_grad()\ndef infer_probs(model, xwins, device):\n    x = xwins.unsqueeze(0).to(device)\n    use_cuda = device != \'cpu\' and str(device).startswith(\'cuda\')\n    if use_cuda:\n        try:\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n        except RuntimeError:\n            torch.cuda.empty_cache()\n            o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n    o = torch.sigmoid(model(x).float())\n    return o[0].cpu().numpy()\n\ndef rankpct(x):\n    order = x.argsort(0).argsort(0).astype(np.float64)\n    return order / max(1, x.shape[0] - 1)\n\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def order_and_meta(sdir):\n        fs = glob.glob(sdir + \'/*.dcm\')\n        recs = []\n        ps_list = []\n        for f in fs:\n            try:\n                h = pydicom.dcmread(f, stop_before_pixels=True)\n                iop = getattr(h, \'ImageOrientationPatient\', None)\n                ipp = getattr(h, \'ImagePositionPatient\', None)\n                if iop is not None and ipp is not None and (len(iop) == 6):\n                    r = np.array(iop[:3], float)\n                    c = np.array(iop[3:], float)\n                    n = np.cross(r, c)\n                    pos = float(np.dot(np.array(ipp, float), n))\n                else:\n                    pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n                ps = getattr(h, \'PixelSpacing\', None)\n                ps = float(ps[0]) if ps is not None else 0.5\n                ps_list.append(ps)\n                recs.append((pos, f, ps))\n            except Exception:\n                recs.append((0.0, f, 0.5))\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return ([(f, ps) for _, f, ps in recs], med_ps)\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape\n        cpx = int(round(CROP_MM / max(ps, 0.001)))\n        cpx = min(cpx, min(h, w))\n        y0 = (h - cpx) // 2\n        x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n    return (order_and_meta, read_px, mm_crop_resize)\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    cands = [r for r in rows if r[\'Anatomical_Plane\'] == plane and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get(\'Fluid_Sensitive\', 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\ndef build_study(sid, ser_records, tsdir, reader):\n    order_and_meta, read_px, mm_crop_resize = reader\n    rows = ser_records.get(sid, [])\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    idx = 0\n    used = set()\n    for plane, fluid, k in SLOTS:\n        r = _pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            idx += k\n            continue\n        used.add(r[\'SeriesInstanceUID\'])\n        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r[\'SeriesInstanceUID\']}")\n        if not files:\n            idx += k\n            continue\n        n = len(files)\n        lo, hi = (int(n * 0.06), int(n * 0.94) - 1)\n        hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n        arrs = []\n        pss = []\n        for p in picks:\n            fp, ps = files[min(p, n - 1)]\n            try:\n                arrs.append(read_px(fp))\n                pss.append(ps)\n            except Exception:\n                arrs.append(None)\n                pss.append(med_ps)\n        valid = [a for a in arrs if a is not None]\n        if valid:\n            allpx = np.concatenate([a.ravel() for a in valid])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = (0.0, 1.0)\n        for a, ps in zip(arrs, pss):\n            if idx >= MAXS:\n                break\n            if a is None:\n                idx += 1\n                continue\n            aw = np.clip((a - loq) / (hiq - loq + 1e-06), 0, 1)\n            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)\n            vol[idx] = (aw * 255).astype(np.uint8)\n            idx += 1\n        if idx >= MAXS:\n            break\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return (vol, mask)\n\ndef find_test_root():\n    cands = [\'/kaggle/input/competitions/rsna-knee-abnormality-detection\', \'/kaggle/input/rsna-knee-abnormality-detection\']\n    for b in cands:\n        if os.path.exists(b + \'/test.csv\'):\n            return b\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f and (os.path.isdir(d + \'/test_series\') or os.path.isdir(d + \'/test_images\')):\n            return d\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f:\n            return d\n    raise RuntimeError(\'no test root under /kaggle/input\')\n\ndef find_weight_file(fname):\n    direct = [f\'/kaggle/input/raptor-knee-arms/{fname}\', f\'/kaggle/input/raptor-knee-arms/1/{fname}\', f\'/kaggle/input/raptor-cnn336/{fname}\']\n    for p in direct:\n        if os.path.exists(p):\n            return p\n    for d in sorted(glob.glob(\'/kaggle/input/*/\')):\n        if \'competition\' in d.lower():\n            continue\n        hits = glob.glob(os.path.join(d, \'**\', fname), recursive=True)\n        if hits:\n            return hits[0]\n    raise RuntimeError(f\'{fname} not found under /kaggle/input\')\n\ndef main():\n    import pandas as pd\n    t0 = time.time()\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    ngpu = torch.cuda.device_count()\n    print(f\'device {dev} | gpus {ngpu} | torch {torch.__version__}\', flush=True)\n    ROOT = find_test_root()\n    tsdir = ROOT + \'/test_series\'\n    if not os.path.isdir(tsdir):\n        tsdir = ROOT + \'/test_images\'\n    print(\'test root:\', ROOT, \'| series dir:\', tsdir, flush=True)\n    test = pd.read_csv(ROOT + \'/test.csv\')\n    test[\'StudyInstanceUID\'] = test[\'StudyInstanceUID\'].astype(str)\n    test_ids = test[\'StudyInstanceUID\'].tolist()\n    tser = pd.read_csv(ROOT + \'/test_series.csv\')\n    tser[\'StudyInstanceUID\'] = tser[\'StudyInstanceUID\'].astype(str)\n    tser[\'SeriesInstanceUID\'] = tser[\'SeriesInstanceUID\'].astype(str)\n    SER = {k: v.to_dict(\'records\') for k, v in tser.groupby(\'StudyInstanceUID\')}\n    print(f\'test studies {len(test_ids)} | test series {len(tser)}\', flush=True)\n    sub_cols = [\'StudyInstanceUID\'] + LAB\n    ssub = os.path.join(ROOT, \'sample_submission.csv\')\n    if os.path.exists(ssub):\n        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)\n    reader = _make_reader()\n    N = len(test_ids)\n    A = len(ARMS)\n    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]\n    for a, arm in enumerate(ARMS):\n        wp = find_weight_file(arm[\'file\'])\n        model, res = load_model(wp, arm[\'arch\'], arm[\'res\'], dev)\n        print(f"[arm {a}] loaded {arm[\'file\']} | res {res} | {time.time() - t0:.0f}s", flush=True)\n        for i, sid in enumerate(test_ids):\n            try:\n                vol, mask = build_study(sid, SER, tsdir, reader)\n                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)\n                arm_probs[a][i] = infer_probs(model, xw, dev)\n                del vol, mask, xw\n            except Exception as e:\n                print(f\'  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})\', flush=True)\n            if (i + 1) % 100 == 0 or i + 1 == N:\n                print(f\'  [arm {a}] {i + 1}/{N} | {time.time() - t0:.0f}s\', flush=True)\n        del model\n        gc.collect()\n        if str(dev).startswith(\'cuda\'):\n            torch.cuda.empty_cache()\n        print(f\'[arm {a}] done + freed | {time.time() - t0:.0f}s\', flush=True)\n    _w = np.array([float(a.get(\'w\', 1.0)) for a in ARMS], dtype=np.float64)\n    _w = _w / _w.sum()\n    print(f"[blend] weighted rank-mean w={dict(zip([a[\'file\'] for a in ARMS], _w.round(4)))}", flush=True)\n    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]), axes=(0, 0))\n    if not np.isfinite(ranks).all():\n        ranks[~np.isfinite(ranks)] = 0.5\n    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)\n    sub.insert(0, \'StudyInstanceUID\', test_ids)\n    sub = sub[sub_cols]\n    assert list(sub.columns) == sub_cols, \'column order drift\'\n    assert sub[\'StudyInstanceUID\'].tolist() == test_ids, \'row identity drift\'\n    assert np.isfinite(sub[LAB].values).all()\n    out = \'/kaggle/working/_raptor.csv\'\n    sub.to_csv(out, index=False)\n    print(\'wrote\', out, \'|\', len(sub), \'rows x\', len(sub.columns), \'cols\', flush=True)\n    print(sub.head().to_string(index=False), flush=True)\n    print(f\'DONE {time.time() - t0:.0f}s\', flush=True)'
WORKER_SRC = 'import os, sys, time, glob\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\'); os.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nt0 = time.time()\nGT = os.environ.get(\'GTAG\', \'?\')\ndef tlog(m): print(f\'[g{GT} t={time.time()-t0:.0f}s] {m}\', flush=True)\n_KE_NS = {\'__name__\': \'_ke\'}\nexec(compile(open(\'/kaggle/working/_ke_src.py\').read(), \'<ke>\', \'exec\'), _KE_NS)\n\n# ===== FASE 1: cache volume uint8 ke disk (dipakai lintas epoch) =====\nimport numpy as np, pandas as pd, os\nKE = _KE_NS\nLAB = KE[\'LAB\']\nROOT = KE[\'find_test_root\']()\ntsdir = ROOT + \'/train_series\'\ntrain = pd.read_csv(ROOT + \'/train.csv\')\ntrain[\'StudyInstanceUID\'] = train[\'StudyInstanceUID\'].astype(str)\ntser = pd.read_csv(ROOT + \'/train_series.csv\')\nfor c in (\'StudyInstanceUID\', \'SeriesInstanceUID\'):\n    tser[c] = tser[c].astype(str)\nSER = {k: v[[\'SeriesInstanceUID\', \'Anatomical_Plane\', \'Fluid_Sensitive\']].to_dict(\'records\')\n       for k, v in tser.groupby(\'StudyInstanceUID\')}\nGOLD = set(train.loc[train[LAB].notna().all(axis=1), \'StudyInstanceUID\'])\nALL_IDS = [s for s in train[\'StudyInstanceUID\'] if s in SER]\ntlog(f\'{len(ALL_IDS)} studi utk cache ({len(GOLD)} gold dikecualikan dr latihan nanti)\')\n\nCDIR = \'/kaggle/tmp/volcache\'\nos.makedirs(CDIR, exist_ok=True)\nimport glob as _g\nMOUNTED = {}\nfor _d in (_g.glob(\'/kaggle/input/*/volcache\') + _g.glob(\'/kaggle/input/*/*/volcache\')\n           + _g.glob(\'/kaggle/input/*/*/*/volcache\')):   # kedalaman terbatas: JANGAN \'**\' (820rb berkas kompetisi)\n    if os.path.isdir(_d):\n        for _f in os.listdir(_d):\n            if _f.endswith(\'.npz\'):\n                MOUNTED[_f[:-4]] = os.path.join(_d, _f)\ntlog(f\'cache termount dari kernel CPU: {len(MOUNTED)} studi (sisanya di-decode di sini)\')\nreader = KE[\'_make_reader\']()\n\ndef cache_one(sid):\n    fn = f\'{CDIR}/{sid}.npz\'\n    if sid in MOUNTED or os.path.exists(fn):\n        return \'skip\'\n    try:\n        vol, mask = KE[\'build_study\'](sid, SER, tsdir, reader)\n        np.savez_compressed(fn, vol=vol, mask=mask)\n        return \'ok\'\n    except Exception as e:\n        return f\'fail:{type(e).__name__}\'\n\nfrom concurrent.futures import ThreadPoolExecutor\nstat = {}\nwith ThreadPoolExecutor(8) as ex:\n    for i, r in enumerate(ex.map(cache_one, ALL_IDS)):\n        stat[r] = stat.get(r, 0) + 1\n        if (i + 1) % 250 == 0:\n            tlog(f\'cache {i+1}/{len(ALL_IDS)} | {stat}\')\ntlog(f\'cache SELESAI {stat}\')\n\n# ===== FASE 2: K-fold CoAtNet -> OOF utk noisy-student (RESEARCH_NOTES §32) =====\nimport numpy as np, pandas as pd, torch, torch.nn.functional as F, gc, math, random, hashlib, os\nimport glob as _glob\n\nDEV = \'cuda\'\nLAB = KE[\'LAB\']; N_T = len(LAB)\nFOLD = int(os.environ[\'FOLD\']); NFOLD = int(os.environ[\'NFOLD\'])\nEPOCHS = int(os.environ[\'EPOCHS\']); INIT = os.environ[\'INIT\']; TAG = os.environ[\'TAG\']\nK_EVAL_OOF = 32\n\ndef _bglob(name):\n    return [p for d in (\'/kaggle/input/*/\', \'/kaggle/input/*/*/\', \'/kaggle/input/*/*/*/\')\n            for p in _glob.glob(d + name)]\n_hits = _bglob(\'fused_labels_r1.csv\')\ntlog(\'label csv: \' + repr(_hits))\nassert _hits, \'blended label csv tidak ketemu di /kaggle/input\'\nfl = pd.read_csv(_hits[0])\nfl[\'StudyInstanceUID\'] = fl[\'StudyInstanceUID\'].astype(str)\nFL = fl.set_index(\'StudyInstanceUID\')\nCONF_COLS = {t: f\'{t}__conf\' for t in LAB}\n\n# ---- fold sadar-GRUP (laporan byte-identik = satu grup), deterministik ----\nweak_ids = [s for s in ALL_IDS if s not in GOLD and s in FL.index]\nrep = train.set_index(\'StudyInstanceUID\')[\'Report\'] if \'Report\' in train.columns else None\ndef gkey(s):\n    r = None if rep is None else rep.get(s)\n    if r is None or isinstance(r, float) or not str(r).strip():\n        return \'u:\' + s\n    return \'r:\' + hashlib.md5(\' \'.join(str(r).split()).encode()).hexdigest()\ngroups = {}\nfor s in weak_ids:\n    groups.setdefault(gkey(s), []).append(s)\nglist = sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0]))\nsizes = [0] * NFOLD; fold_of = {}\nfor gk, members in glist:\n    f = int(np.argmin(sizes))\n    for m in members:\n        fold_of[m] = f\n    sizes[f] += len(members)\ntlog(f\'grup={len(groups)} studi_lemah={len(weak_ids)} ukuran_fold={sizes}\')\ntrain_ids = [s for s in weak_ids if fold_of[s] != FOLD]\nheld_ids = [s for s in weak_ids if fold_of[s] == FOLD]\ngold_ids = sorted(GOLD)\ntlog(f\'fold {FOLD}: latih={len(train_ids)} held-out={len(held_ids)} gold={len(gold_ids)}\')\n# keamanan: tak boleh ada laporan kembar lintas latih/held-out\n_tg = {gkey(s) for s in train_ids}; _hg = {gkey(s) for s in held_ids}\nassert not (_tg & _hg), \'BOCOR: ada grup laporan yang sama di latih & held-out\'\n\ndef load_study(sid):\n    fn = f\'/kaggle/tmp/volcache/{sid}.npz\'\n    if not os.path.exists(fn):\n        fn = MOUNTED[sid]\n    z = np.load(fn)\n    return z[\'vol\'], z[\'mask\']\n\ndef rand_windows(vol, mask, k, res):\n    D = vol.shape[0]\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = int(valid.min()), int(valid.max())\n    cs = [c for c in range(max(1, lo), min(D - 1, hi))]\n    if len(cs) == 0: cs = [max(1, min(D - 2, (lo + hi) // 2))]\n    cs = list(np.random.choice(cs, size=min(k, len(cs)), replace=False))\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    MEAN = KE[\'_MEAN\']; STD = KE[\'_STD\']\n    for j, c in enumerate(cs):\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n        if random.random() < 0.5:\n            tri *= random.uniform(0.9, 1.1)\n        t = torch.from_numpy(tri)\n        if t.shape[-1] != res:\n            import torch.nn.functional as _F\n            t = _F.interpolate(t[None], size=(res, res), mode=\'bilinear\', align_corners=False)[0]\n        wins[j] = ((t - MEAN) / STD).numpy()\n    return torch.from_numpy(wins)\n\nK_WIN = 12\nACC_STUDIES = 8\nLR_HEAD, LR_BB, WD = 1e-3, 2e-5, 0.05\nWARMUP = 150\n\n_CK = {\'c5\': \'raptor_ft_coatnet_v5_full_swa.pt\', \'v10\': \'raptor_ft_coatnet_v10_full.pt\', \'v8\': \'raptor_ft_coatnet_v8_full_swa.pt\'}\nwp = KE[\'find_weight_file\'](_CK[INIT])\nmodel, RES = KE[\'load_model\'](wp, \'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', 384, DEV)\n\ndef predict_ids(ids, k=K_EVAL_OOF):\n    model.eval(); out = []\n    with torch.no_grad():\n        for sid in ids:\n            try:\n                v, m = load_study(sid)\n                xw = KE[\'eval_windows\'](v, m, k=k, res=RES, norm=KE[\'NORM\'])\n                out.append(KE[\'infer_probs\'](model, xw, DEV))\n            except Exception as e:\n                out.append(np.full(N_T, 0.5, np.float32))\n    return np.stack(out)\n\ndef corr_rows(P, ids, tag):\n    """korelasi Pearson rata-rata (lintas 12 label) antara prediksi dan label guru."""\n    T = FL.loc[ids, LAB].fillna(0.28).values.astype(float)\n    cs = [np.corrcoef(P[:, j], T[:, j])[0, 1] for j in range(N_T)]\n    tlog(f\'   korelasi(pred, guru) [{tag}] rata2={np.nanmean(cs):.4f}\')\n    return float(np.nanmean(cs))\n\n# ---- PROBE kontaminasi: model inisial pada studi lemah held-out vs gold (tak pernah dilihat) ----\nrs = np.random.RandomState(7)\nprobe_ids = [held_ids[i] for i in rs.permutation(len(held_ids))[:150]]\nPp = predict_ids(probe_ids)\nPg0 = predict_ids(gold_ids)\npd.DataFrame(Pp, columns=LAB).assign(StudyInstanceUID=probe_ids).to_csv(\n    f\'/kaggle/working/probe_{TAG}_{INIT}_weak_f{FOLD}.csv\', index=False)\npd.DataFrame(Pg0, columns=LAB).assign(StudyInstanceUID=gold_ids).to_csv(\n    f\'/kaggle/working/probe_{TAG}_{INIT}_gold_f{FOLD}.csv\', index=False)\ntlog(\'PROBE (model inisial, sebelum latih):\')\nc_seen = corr_rows(Pp, probe_ids, \'studi lemah = DILIHAT c5 saat pra-latih\')\n# guru utk gold: pakai label blend Claude-pilot+flight tidak tersedia di kernel -> pakai label\n# gold asli sbg acuan alternatif utk perbandingan korelasi terhadap KEBENARAN\nYg = train.set_index(\'StudyInstanceUID\').loc[gold_ids, LAB].values.astype(float)\ncg = [np.corrcoef(Pg0[:, j], Yg[:, j])[0, 1] for j in range(N_T)]\ntlog(f\'   korelasi(pred, GOLD asli) [gold = TAK pernah dilihat] rata2={np.nanmean(cg):.4f}\')\n\nmodel.backbone.requires_grad_(True)\nhead_params = list(model.norm.parameters()) + list(model.att.parameters()) + [model.clsW, model.clsb]\nbb_params = [p for p in model.backbone.parameters() if p.requires_grad]\nopt = torch.optim.AdamW([{\'params\': bb_params, \'lr\': LR_BB},\n                         {\'params\': head_params, \'lr\': LR_HEAD}], weight_decay=WD)\ntotal_steps = max(1, (len(train_ids) * EPOCHS) // ACC_STUDIES)\nsched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP) *\n        0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps))))\nscaler = torch.amp.GradScaler(\'cuda\')\ntlog(f\'model siap res={RES} | steps={total_steps} | FULL FT bb lr={LR_BB}\')\n\ny_lab = FL.loc[train_ids, LAB].fillna(0.28).values.astype(np.float32)\nc_lab = np.stack([FL.loc[train_ids, CONF_COLS[t]].fillna(0.15).values.astype(np.float32)\n                  for t in LAB], 1)\nW_CONF = 0.5 + np.clip(c_lab, 0, 1)\n\nfor ep in range(EPOCHS):\n    order = np.random.permutation(len(train_ids))\n    model.train()\n    opt.zero_grad(set_to_none=True)\n    ep_loss, nacc = 0.0, 0\n    for ii, si in enumerate(order):\n        sid = train_ids[si]\n        try:\n            vol, mask = load_study(sid)\n            xw = rand_windows(vol, mask, K_WIN, RES)\n            xb = xw.unsqueeze(0).to(DEV)\n            yb = torch.from_numpy(y_lab[si])[None].to(DEV)\n            wb = torch.from_numpy(W_CONF[si])[None].to(DEV)\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                logits = model(xb).float()\n                loss = (F.binary_cross_entropy_with_logits(logits, yb, reduction=\'none\') * wb).mean()\n            scaler.scale(loss).backward()\n            ep_loss += float(loss.detach())\n            del vol, mask, xw\n        except torch.cuda.OutOfMemoryError:\n            torch.cuda.empty_cache()\n            tlog(f\'OOM di {sid[:16]} -- skip studi ini\')\n            continue\n        nacc += 1\n        if nacc % ACC_STUDIES == 0 or ii == len(order) - 1:\n            scaler.step(opt); scaler.update(); sched.step()\n            opt.zero_grad(set_to_none=True)\n        if (ii + 1) % 400 == 0:\n            tlog(f\'ep{ep+1} {ii+1}/{len(order)} loss~{ep_loss/(ii+1):.4f} \'\n                 f\'lr_bb={sched.get_last_lr()[0]:.2e}\')\n    tlog(f\'=== EPOCH {ep+1}/{EPOCHS} selesai | loss={ep_loss/max(1,len(order)):.4f}\')\n\ntorch.save({\'arch\': \'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', \'res\': RES,\n            \'model\': model.state_dict()}, f\'/kaggle/working/kfold{TAG}_{INIT}_f{FOLD}.pt\')\ntlog(\'checkpoint fold disimpan\')\n\n# ---- OOF: prediksi studi held-out (TAK dilihat model ini) + gold-58 ----\nPh = predict_ids(held_ids)\nPg = predict_ids(gold_ids)\noof = pd.DataFrame(np.vstack([Ph, Pg]), columns=LAB)\noof.insert(0, \'StudyInstanceUID\', held_ids + gold_ids)\noof[\'is_gold\'] = [0] * len(held_ids) + [1] * len(gold_ids)\noof[\'fold\'] = [FOLD] * len(held_ids) + [-1] * len(gold_ids)\noof.to_csv(f\'/kaggle/working/oof_{TAG}_{INIT}_f{FOLD}.csv\', index=False)\ntlog(f\'OOF ditulis: {len(held_ids)} held-out + {len(gold_ids)} gold\')\n\ndef auc1(yy, ss):\n    pos, neg = ss[yy == 1], ss[yy == 0]\n    if len(pos) == 0 or len(neg) == 0: return np.nan\n    r = pd.Series(np.concatenate([pos, neg])).rank().values\n    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))\nytrue = train.set_index(\'StudyInstanceUID\').loc[gold_ids, LAB].values.astype(float)\naucs = [auc1(ytrue[:, j], Pg[:, j]) for j in range(N_T)]\ntlog(f\'MACRO AUC gold-58 (fold {FOLD}, {INIT}-init): {np.nanmean(aucs):.4f}  \'\n     f\'[acuan v3=0.9234 (LB 0.932), c5=0.9213]\')\ntlog(\'PASCA-LATIH:\')\ncorr_rows(Ph, held_ids, \'studi lemah HELD-OUT (tak dilihat fine-tune; c5 pra-latih melihatnya)\')\ncg2 = [np.corrcoef(Pg[:, j], ytrue[:, j])[0, 1] for j in range(N_T)]\ntlog(f\'   korelasi(pred, GOLD asli) [gold] rata2={np.nanmean(cg2):.4f}\')\ntlog(\'KFOLD DONE\')\n'
FOLDS = [0, 1]
NFOLD = 4
INIT = 'c5'
LABELS = 'r1'
TAG = 'r2'
EPOCHS = 3


In [ ]:

import os, sys, time, json, subprocess, glob
import numpy as np, pandas as pd
t0 = time.time()
def tlog(m): print(f'[main t={time.time()-t0:.0f}s] {m}', flush=True)
open('/kaggle/working/_ke_src.py', 'w').write(KE_SRC)
open('/kaggle/working/worker.py', 'w').write(WORKER_SRC)
import torch
NG = max(1, torch.cuda.device_count())
tlog(f'GPU: {NG} | folds {FOLDS} | init={INIT} labels={LABELS} tag={TAG} epochs={EPOCHS}')
procs = {}
for i, f in enumerate(FOLDS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % NG), FOLD=str(f), NFOLD=str(NFOLD), INIT=INIT, EPOCHS=str(EPOCHS), TAG=TAG, GTAG=f'{i%NG}f{f}')
    fh = open(f'/kaggle/working/worker_f{f}.log', 'w')
    procs[f] = (subprocess.Popen([sys.executable, '-u', '/kaggle/working/worker.py'], env=env, stdout=fh, stderr=subprocess.STDOUT), fh)
pos = {f: 0 for f in procs}
def pump():
    for f in procs:
        with open(f'/kaggle/working/worker_f{f}.log') as fh:
            fh.seek(pos[f]); new = fh.read(); pos[f] = fh.tell()
        for line in new.splitlines(): print(line, flush=True)
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(10); pump()
pump()
for f, (p, fh) in procs.items(): tlog(f'fold {f} rc={p.returncode}')
tlog('SELESAI')
